# 09 - Hourly/monthly post-processing, normalization, and plots

This notebook performs post-processing over previously generated outputs, including normalization, time aggregation, and plot generation.

In practical terms, we will move through preparation, configuration, execution, and verification so every transition in the workflow stays explicit.
As the notebook progresses, each code cell is introduced with context first, then the call itself, and finally quick guidance on how to interpret the result.

Workflow roadmap:
- Preparation: imports, path configuration, and baseline model loading.
- Configuration: output selection, parameter definitions, and problem setup.
- Execution and review: simulation or optimization calls followed by targeted checks.

Main classes, methods, and functions used in this notebook:
- `ParametricSimulation.load_outputs_parametric`
- `ParametricSimulation.normalize_outputs`
- `OptimisationSimulation.load_outputs_optimisation`
- `OptimisationSimulation.plot_parallel_coordinates`
- `FileNotFoundError`
- `ParametricSimulation.__new__`
- `parametric.load_outputs_parametric`
- `parametric.normalize_outputs`
- `parametric.get_hourly_df`
- `parametric.get_monthly_df`

Narrative structure used in this notebook:
- Each code cell is preceded by an explanatory markdown block.
- Section headings are contextual to the current notebook and execution block.
- Key arguments are described before they are used in code.
- Execution-oriented cells may include a follow-up markdown note to help interpret results.

## Post-processing: import workflow dependencies

We start by importing the APIs required for this notebook block.

Keeping imports explicit at the top of each topic notebook makes selective reruns safer when you jump between sections of the series.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [ ]:
from pathlib import Path

from accim.parametric_and_optimisation.main import ParametricSimulation, OptimisationSimulation


## Post-processing: locate latest output files

This cell locates the most recent result files that will be reused for downstream analysis.

By resolving file paths here, the notebook stays reproducible and avoids accidentally mixing outputs from different runs.

Key arguments used in this cell and why they matter:
- `parents`: defines this setting for the current call.
- `exist_ok`: defines this setting for the current call.
- `key`: defines this setting for the current call.

Variables created in this cell:
- `PROJECT_ROOT`: reused in subsequent workflow blocks.
- `PLOTS_OUT_DIR`: reused in subsequent workflow blocks.
- `parametric_candidates`: reused in subsequent workflow blocks.
- `optim_candidates`: reused in subsequent workflow blocks.
- `parametric_csv`: reused in subsequent workflow blocks.
- `optim_csv`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `PROJECT_ROOT`, `PLOTS_OUT_DIR`, `parametric_candidates`, `optim_candidates`.

In [ ]:

PROJECT_ROOT = Path.cwd()
PLOTS_OUT_DIR = PROJECT_ROOT / "nb09_postprocessing_plots"
PLOTS_OUT_DIR.mkdir(parents=True, exist_ok=True)

parametric_candidates = sorted(
    PROJECT_ROOT.glob("**/outputs_param_simulation_*.csv"),
    key=lambda p: p.stat().st_mtime,
)
optim_candidates = sorted(
    PROJECT_ROOT.glob("**/outputs_optimisation_*.csv"),
    key=lambda p: p.stat().st_mtime,
)

if not parametric_candidates:
    raise FileNotFoundError(
        "No parametric simulation CSV files were found. Run notebooks 01/02/03/05/06 first."
    )
if not optim_candidates:
    raise FileNotFoundError(
        "No optimisation CSV files were found. Run notebooks 07/08 first."
    )

parametric_csv = parametric_candidates[-1]
optim_csv = optim_candidates[-1]

print("Latest parametric CSV:", parametric_csv)
print("Latest optimisation CSV:", optim_csv)


## Post-processing: load parametric outputs, normalize, and generate baseline plots

Here we normalize loaded outputs and prepare comparable analysis-ready views.

Applying normalization consistently across notebooks is key if you later compare plots and metrics between workflows.

Key arguments used in this cell and why they matter:
- `csv_path`: path to the output CSV file to load.
- `df_types`: dataset groups that should be normalized.
- `start_date`: start timestamp used for time indexing.
- `normalize_per_m2`: whether results are normalized by floor area.
- `df_source`: defines this setting for the current call.
- `col`: facet column used in plotting.
- `out_dir`: output folder for generated artifacts and results.

Variables created in this cell:
- `parametric`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm normalization scope (`df_types`) matches the intended datasets.
- Check one or two normalized columns to verify value scaling behaves as expected.

In [ ]:

parametric = ParametricSimulation.__new__(ParametricSimulation)
parametric.last_run_type = None
parametric.outputs_param_simulation = None
parametric.outputs_param_simulation_hourly = None
parametric.outputs_param_simulation_monthly = None
parametric.outputs_param_simulation_filepath = None
parametric.outputs_normalized = False
parametric.epws = []
parametric.buildings = []

parametric.load_outputs_parametric(csv_path=str(parametric_csv))
print("Loaded parametric rows:", len(parametric.outputs_param_simulation))

parametric.normalize_outputs(df_types=["parametric"])
print("Parametric normalization completed.")

parametric.get_hourly_df(start_date="2024-01-01 01", normalize_per_m2=False)
print("Parametric hourly shape:", parametric.outputs_param_simulation_hourly.shape)

parametric.get_monthly_df(start_date="2024-01-01 01", normalize_per_m2=False)
print("Parametric monthly shape:", parametric.outputs_param_simulation_monthly.shape)

parametric.plot_categorical_boxplots(
    df_source="parametric",
    col="epw",
    out_dir=str(PLOTS_OUT_DIR),
    normalize_per_m2=False,
)
print("Parametric boxplots generated.")


## Post-processing: prepare hourly dataframe and plotting configuration

Create figures for interpretation and reporting.

This section converts tabular outputs into visual diagnostics for interpretation. Verify axes, filters, and grouping dimensions to ensure valid comparisons.

Key arguments used in this cell and why they matter:
- `csv_path`: path to the output CSV file to load.
- `only_pareto_optimal`: whether analysis is restricted to non-dominated rows.
- `skip_confirmation`: skips interactive confirmation prompts.
- `start_date`: start timestamp used for time indexing.
- `normalize_per_m2`: whether results are normalized by floor area.
- `out_dir`: output folder for generated artifacts and results.

Variables created in this cell:
- `optim`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:

optim = OptimisationSimulation.__new__(OptimisationSimulation)
optim.last_run_type = None
optim.outputs_optimisation = None
optim.outputs_optimisation_filepath = None
optim.outputs_optimisation_hourly = None
optim.outputs_optimisation_monthly = None
optim.outputs_normalized = False
optim.optimisation_csv_paths_non_dominated = []
optim.optimisation_csv_paths_dominated = []
optim.optimisation_csv_paths_non_dominated_by_epw = {}
optim.optimisation_csv_paths_dominated_by_epw = {}
optim.evaluators = {}
optim.epws = []
optim.buildings = []

optim.load_outputs_optimisation(csv_path=str(optim_csv))
print("Loaded optimisation rows:", len(optim.outputs_optimisation))

optim.get_hourly_df_optimisation(
    only_pareto_optimal=True,
    skip_confirmation=True,
    start_date="2024-01-01 01",
    normalize_per_m2=False,
)
print("Optimisation hourly shape:", optim.outputs_optimisation_hourly.shape)

optim.get_monthly_df_optimisation(start_date="2024-01-01 01", normalize_per_m2=False)
print("Optimisation monthly shape:", optim.outputs_optimisation_monthly.shape)

optim.plot_parallel_coordinates(out_dir=str(PLOTS_OUT_DIR))
print("Parallel coordinates plot generated.")

optim.plot_pairwise_scatter_matrix(out_dir=str(PLOTS_OUT_DIR), normalize_per_m2=False)
print("Pairwise scatter matrix generated.")


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.